### Uploading Stata `.dta` Files from Autor, Dorn, Hanson and Creating Silver Spark Tables

In [0]:
# load raw STATA dataset from david dorn's data files:
import pandas as pd
trade_df = pd.read_stata("/Volumes/workspace/default/china_trade_data/sic87dd_trade_data.dta")

In [0]:
# convert the pandas DF to spark format:
spark_df = spark.createDataFrame(trade_df)
display(spark_df)

### Data Profiling:

In [0]:
# check for null values in each column:
from pyspark.sql.functions import col, sum
null_counts = spark_df.select(
    [sum(col(c).isNull().cast('int')).alias(c) for c in spark_df.columns]
)
display(null_counts)

In [0]:
# profile the importer column:
display(spark_df.select('importer').distinct())

### Practicing Data Aggregation/Validation:

In [0]:
# practice spark data manipulation syntax:
display(
    spark_df
    .filter(spark_df.importer == 'USA')
    .groupBy('exporter')
    .sum('imports')
    .orderBy('sum(imports)', ascending=False)
)

### Create a Silver Delta Table with Spark:

In [0]:
# Create a new schema for the project:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.china_shock")

# Create a silver delta table:
spark_df.write.format('delta').mode('overwrite').saveAsTable('workspace.china_shock.trade_silver')


workspace